<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/07-trigger-types.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 14.7 — Trigger types: cadence is a deliberate choice

Compare the default trigger's batch spacing against `ProcessingTime`
via `query.recentProgress` timestamps, then run `Trigger.AvailableNow`
against a fixed backlog of files and watch the query terminate itself.

In [ ]:
import os, sys, shutil, time, json
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.7-trigger-types")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Default trigger: back-to-back batches, timed via `recentProgress`

In [ ]:
default_q = (spark.readStream.format("rate").option("rowsPerSecond", 20).load()
            .writeStream.format("memory").queryName("default_trigger_demo")
            .outputMode("append").start())

time.sleep(5)
default_q.stop()

progress = default_q.recentProgress
gaps = []
for i in range(1, len(progress)):
    import datetime
    t0 = datetime.datetime.fromisoformat(progress[i-1]["timestamp"].replace("Z", "+00:00"))
    t1 = datetime.datetime.fromisoformat(progress[i]["timestamp"].replace("Z", "+00:00"))
    gaps.append((t1 - t0).total_seconds())
print(f"default trigger -- {len(progress)} batches, gaps between batch starts (seconds): {gaps}")
print("-> back-to-back: gaps should be small/near-zero, next batch starts as soon as the last finishes")

## `ProcessingTime` — a fixed cadence, wider gaps

In [ ]:
pt_q = (spark.readStream.format("rate").option("rowsPerSecond", 20).load()
        .writeStream.trigger(processingTime="2 seconds")
        .format("memory").queryName("pt_trigger_demo")
        .outputMode("append").start())

time.sleep(9)
pt_q.stop()

progress = pt_q.recentProgress
gaps = []
for i in range(1, len(progress)):
    import datetime
    t0 = datetime.datetime.fromisoformat(progress[i-1]["timestamp"].replace("Z", "+00:00"))
    t1 = datetime.datetime.fromisoformat(progress[i]["timestamp"].replace("Z", "+00:00"))
    gaps.append(round((t1 - t0).total_seconds(), 1))
print(f"processingTime='2 seconds' -- gaps between batch starts: {gaps}")
print("-> should cluster near 2.0s apart, unlike the default trigger's near-zero gaps")

## `Trigger.AvailableNow` — processes the backlog, then STOPS ITSELF

In [ ]:
shutil.rmtree("stream_in", ignore_errors=True)
os.makedirs("stream_in", exist_ok=True)
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([StructField("msg", StringType())])

for i in range(5):
    with open(f"stream_in/f{i}.json", "w") as f:
        f.write(json.dumps({"msg": f"hello-{i}"}) + "\n")

an_q = (spark.readStream.schema(schema).json("stream_in/")
        .writeStream.trigger(availableNow=True)
        .format("memory").queryName("availablenow_demo")
        .option("checkpointLocation", "checkpoints/an_demo")
        .outputMode("append").start())

an_q.awaitTermination()   # returns on its own -- no manual .stop() needed!
print("query stopped itself:", not an_q.isActive)
spark.sql("SELECT * FROM availablenow_demo").show()

## `Trigger.Continuous` — mentioned, not exercised

Experimental, narrow operator support (row-level only — no
aggregation, no stateful ops, no stream-stream joins from this
module). Shown as reference, not run.

In [ ]:
# query = (df.writeStream
#           .trigger(continuous="1 second")   # EXPERIMENTAL -- sub-second latency
#           .format("console")
#           .start())
print("Trigger.Continuous supports only row-level transforms (map/filter) --")
print("everything from 14.3 (watermarks) through 14.5 (stream joins) needs the micro-batch engine.")

In [ ]:
shutil.rmtree("stream_in", ignore_errors=True)
shutil.rmtree("checkpoints", ignore_errors=True)
spark.stop()

## Exercises

1. Change `processingTime` to `"500 milliseconds"` and re-measure the
   gaps. At very short intervals, does the observed cadence still
   match the requested one, or does actual batch processing time start
   to dominate (reread the lesson's "if a batch takes longer..."
   note)?
2. Add 5 MORE files to `stream_in/` after the `AvailableNow` query has
   already terminated, then start a SECOND `AvailableNow` query with
   the SAME checkpoint directory. Does it reprocess the first 5, or
   only pick up the new ones? Connect this to 14.6.
3. Run the `AvailableNow` demo with only 1 file instead of 5. Does it
   still make sense as "streaming code with batch semantics," or does
   it start to look just like an ordinary batch read? What's still
   different under the hood even with one file?
4. For the default-trigger demo, increase `rowsPerSecond` by 10x. Do
   the gaps between batches grow, shrink, or stay about the same?
   What does this tell you about what "back-to-back" actually depends
   on?
5. Sketch (no need to run) how you'd schedule the `AvailableNow` query
   from this notebook as a nightly cron/Airflow job (Module 17) instead
   of a long-running process — what changes about how you invoke the
   script, and what stays exactly the same in the query code itself?

In [ ]:
# your exercise code here